# ESE 670 — Homework 3: one diffusion model, three samplers

**Data.** A mixture of eight Gaussians in the plane. For this data the noisy densities
$p_\tau$, their scores and the ideal denoiser are known **in closed form**. So everything the network learns can be compared
with the truth, and every sampler can be checked against exact samples.

**Goal.** Train one $\varepsilon$-prediction network and read it in four ways: noise $\varepsilon_\theta$, score $s_\theta$,
denoiser $\hat x_0$, velocity $v_\theta$. Then generate samples in three ways and watch individual particles:

* the **reverse SDE** (DDPM ancestral sampling is its discretization): random trajectories;
* the **probability-flow ODE**: deterministic trajectories with the same marginals;
* **DDIM**: large deterministic jumps that follow the ODE.

Finally we **steer** the samplers toward one class: classifier guidance (with the exact Bayes classifier and with a trained noisy
classifier) and classifier-free guidance.

**What to hand in.** The executed notebook with an answer under every **Question 1–5** (short: a formula, a sentence, a number).
All the code is provided: run it, read it, and use its output in your answers.

The **Remarks** are worked explanations: read them, nothing to hand in.

| item | points |
|---|---|
| Question 1 | 2 |
| Question 2 | 2 |
| Question 3 | 2 |
| Question 4 | 2 |
| Question 5 | 2 |
| **Total** | **10** |

Everything runs on a CPU in a few minutes (but you can use Colab GPU).

## 0. Setup and notation

We use **continuous time** $\tau\in[0,1]$ and the variance-preserving (VP) forward SDE of the lecture,
$$\mathrm dx=-\tfrac12\beta(\tau)\,x\,\mathrm d\tau+\sqrt{\beta(\tau)}\,\mathrm dW,\qquad \beta(\tau)=\beta_0+(\beta_1-\beta_0)\tau,\quad \beta_0=0.1,\ \beta_1=20 .$$
Its transition from the data is Gaussian:
$$x_\tau=\sqrt{\bar\alpha(\tau)}\,x_0+\sqrt{1-\bar\alpha(\tau)}\,\varepsilon,\qquad \bar\alpha(\tau)=\exp\Big(-\int_0^\tau\beta(s)\,\mathrm ds\Big)=\exp\Big(-\beta_0\tau-\tfrac12(\beta_1-\beta_0)\tau^2\Big).$$
The discrete DDPM chain with $T=1000$ and $\beta_t$ linear from $10^{-4}$ to $0.02$ is this SDE on the grid $\tau=t/T$
(since $\beta_t\approx\beta(t/T)/T$). Working in continuous time lets the same network be used by every sampler on any time grid.

We never go all the way to $\tau=0$ (the score of the data is singular there); the smallest time is $\tau_{\min}=10^{-3}$.

In [ ]:
import math, copy, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
torch.set_default_dtype(torch.float32)
device = torch.device("cpu")      # feel free to switch to GPU if needed

B0, B1 = 0.1, 20.0
TAU_MIN = 1e-3

# the data of the lecture notebook: 8 Gaussians with std 0.2 on a circle of radius 1.7
r = 1 / math.sqrt(2)
CENTERS = 1.7 * torch.tensor([(1, 0), (-1, 0), (0, 1), (0, -1), (r, r), (r, -r), (-r, r), (-r, -r)], dtype=torch.float32)
STD = 0.2
K = len(CENTERS)

def sample_mixture(n):
    idx = torch.randint(K, (n,))
    return CENTERS[idx] + STD * torch.randn(n, 2)

def tcol(t, x):
    """Time as a column (B, 1) matching the batch x. t may be a float, a 0-d tensor or a (B,) tensor."""
    t = torch.as_tensor(t, dtype=x.dtype)
    if t.ndim == 0:
        t = t.expand(x.shape[0])
    return t.reshape(-1, 1)

def beta(t):
    return B0 + (B1 - B0) * t

## 1. The forward process

We implement `alpha_bar` and `q_sample` (one line each).

In [ ]:
def alpha_bar(t):
    """bar alpha(tau) = exp(-int_0^tau beta)."""
    return torch.exp(-(B0 * t + 0.5 * (B1 - B0) * t ** 2))

def q_sample(x0, t):
    """Sample x_tau ~ q(x_tau | x_0). t has shape (B, 1). Returns (x_tau, eps)."""
    eps = torch.randn_like(x0)
    a = alpha_bar(t)
    x_t = a.sqrt() * x0 + (1 - a).sqrt() * eps
    return x_t, eps

print("alpha_bar(1) =", float(alpha_bar(torch.tensor(1.0))), "  (should be about 4.3e-05)")

In [ ]:
x0 = sample_mixture(3000)
taus = [0.0, 0.05, 0.15, 0.3, 0.6, 1.0]
fig, ax = plt.subplots(1, len(taus), figsize=(16, 2.9))
for a_, tau in zip(ax, taus):
    xt = x0 if tau == 0 else q_sample(x0, tcol(tau, x0))[0]
    a_.scatter(*xt.T, s=1, alpha=0.4)
    a_.set(title=f"tau = {tau}", xlim=(-3.5, 3.5), ylim=(-3.5, 3.5), xticks=[], yticks=[]); a_.set_aspect("equal")
plt.tight_layout(); plt.show()

### Paths in $(\tau,x_1,x_2)$ with the sampled marginals
Throughout the notebook trajectories are also drawn in 3D with time as the horizontal axis. The faint clouds are **exact samples of the
marginals** $p_\tau$ at a few times (data at $\tau=0$, almost $\mathcal N(0,I)$ at $\tau=1$): a correct sampler must carry particles from
cloud to cloud. Below: the forward process itself, simulated with its exact Markov transitions
$x_{\tau'}=\sqrt{\bar\alpha(\tau')/\bar\alpha(\tau)}\,x_\tau+\sqrt{1-\bar\alpha(\tau')/\bar\alpha(\tau)}\,z$.

In [ ]:
MARGINAL_SLICES = (1.0, 0.5, 0.2, 0.0)

def sample_marginal(n, tau):
    x0 = sample_mixture(n)
    return x0 if tau == 0 else q_sample(x0, tcol(tau, x0))[0]

def forward_paths(x0, N=200):
    """Exact forward trajectories on a uniform grid from tau = 0 to 1."""
    g = np.linspace(0.0, 1.0, N + 1)
    x, P = x0, [x0]
    for s_, t_ in zip(g[:-1], g[1:]):
        r = alpha_bar(torch.tensor(t_)) / alpha_bar(torch.tensor(s_))
        x = r.sqrt() * x + (1 - r).sqrt() * torch.randn_like(x)
        P.append(x)
    return torch.stack(P), g

def plot_paths_3d(ax, P, g, title, slices=MARGINAL_SLICES, n_marg=600, end_cloud=None, highlight=None,
                  colors=None, lw=0.8, start_label="start", end_label="end"):
    """P: (L, n, 2) paths at the times g (length L). Faint clouds: exact samples of p_tau at `slices`
    (or `end_cloud` at tau = 0 if given). `highlight`: an extra cloud at tau = 0 drawn in orange (e.g. a target class)."""
    P = np.asarray(P); g = np.asarray(g)
    for k, s_ in enumerate(slices):
        cl = np.asarray(end_cloud) if (s_ == 0 and end_cloud is not None) else np.asarray(sample_marginal(n_marg, s_))
        ax.scatter(np.full(len(cl), s_), cl[:, 0], cl[:, 1], c="steelblue", alpha=0.3, s=6,
                   label="sampled marginals $p_\\tau$" if k == 0 else None)
    if highlight is not None:
        h = np.asarray(highlight)
        ax.scatter(np.zeros(len(h)), h[:, 0], h[:, 1], c="orange", alpha=0.25, s=4, label="target class")
    for i in range(P.shape[1]):
        ax.plot(g, P[:, i, 0], P[:, i, 1], alpha=0.75, lw=lw, color=None if colors is None else colors[i % len(colors)])
    ax.scatter(np.full(P.shape[1], g[0]), P[0, :, 0], P[0, :, 1], c="blue", s=14, label=start_label)
    ax.scatter(np.full(P.shape[1], g[-1]), P[-1, :, 0], P[-1, :, 1], c="red", s=14, label=end_label)
    ax.set_xlabel("time $\\tau$"); ax.set_ylabel("$x_1$"); ax.set_zlabel("$x_2$"); ax.set_title(title, fontsize=10)
    ax.view_init(elev=30, azim=-60); ax.legend(fontsize=7, loc="upper left")

Pf, gf_fwd = forward_paths(sample_mixture(40))
fig = plt.figure(figsize=(12, 6))
plot_paths_3d(fig.add_subplot(111, projection="3d"), Pf, gf_fwd, "forward process: data (tau = 0) to noise (tau = 1)",
              start_label="data x_0", end_label="x_1")
plt.tight_layout(); plt.show()

**Question 1 (2 pts).** Each component of the mixture $\mathcal N(c_k,s^2I)$ of the data is mapped by the forward process to a Gaussian.
We derive $p_\tau$ (the density of $x_\tau$) exactly.
Given $x_0=c_k+s\xi$, $x_\tau=\sqrt{\bar\alpha}\, x_0 + \sqrt{1-\bar\alpha}\,\varepsilon= \sqrt{\bar\alpha}\,c_k+\sqrt{\bar\alpha}\,s\xi+\sqrt{1-\bar\alpha}\,\varepsilon$, a sum of independent
Gaussians. So
$$p_\tau(x)=\frac1K\sum_{k=1}^K\mathcal N\big(x;\ \sqrt{\bar\alpha}\,c_k,\ v_\tau I\big),\qquad v_\tau=\bar\alpha s^2+1-\bar\alpha .$$

Why is $p_1$ indistinguishable from $\mathcal N(0,I)$?

*Your answer:*

## 2. Ground truth: exact score, noise and denoiser

Because $p_\tau$ is a Gaussian mixture, its **score** $\nabla\log p_\tau$ is explicit. With the posterior weights of the
components
$$w_k(x)=\frac{\mathcal N(x;\sqrt{\bar\alpha}c_k,v_\tau I)}{\sum_j\mathcal N(x;\sqrt{\bar\alpha}c_j,v_\tau I)}\qquad(\text{a softmax of }-\|x-\sqrt{\bar\alpha}c_k\|^2/2v_\tau),$$
$$\nabla\log p_\tau(x)=\sum_k w_k(x)\,\frac{\sqrt{\bar\alpha}\,c_k-x}{v_\tau}.$$
From the lecture: the ideal noise prediction is $\varepsilon^\ast(x,\tau)=\mathbb E[\varepsilon\mid x_\tau=x]=-\sqrt{1-\bar\alpha}\,\nabla\log p_\tau(x)$.

We complete `true_score`. The cell after it checks it against automatic differentiation of `true_log_p`.

In [ ]:
def _mixture_terms(x, t):
    a = alpha_bar(tcol(t, x))                         # (B, 1)
    v = a * STD ** 2 + 1 - a                          # (B, 1)
    m = a.sqrt()[:, :, None] * CENTERS[None]          # (B, K, 2): component means sqrt(abar) c_k
    d2 = ((x[:, None, :] - m) ** 2).sum(-1)           # (B, K): squared distances
    return a, v, m, d2

def true_log_p(x, t):
    a, v, m, d2 = _mixture_terms(x, t)
    log_comp = -d2 / (2 * v) - torch.log(2 * math.pi * v)      # log N(x; m_k, v I) in 2D
    return torch.logsumexp(log_comp, dim=1) - math.log(K)

def true_score(x, t):
    a, v, m, d2 = _mixture_terms(x, t)
    w = torch.softmax(-d2 / (2 * v), dim=1)                    # (B, K)
    return (w[:, :, None] * (m - x[:, None, :])).sum(1) / v

def true_eps(x, t):
    return -(1 - alpha_bar(tcol(t, x))).sqrt() * true_score(x, t)

# check against autograd
x = torch.randn(5, 2, requires_grad=True)
g = torch.autograd.grad(true_log_p(x, 0.3).sum(), x)[0]
print("max |closed form - autograd| =", float((g - true_score(x.detach(), 0.3)).abs().max()))

## 3. One network, four readings

All the quantities of the lecture are linear functions of each other, given $x$ and $\tau$:

| quantity | formula_________________________________________ |
|---|---|
| noise prediction | $\varepsilon_\theta(x,\tau)$ (what the network outputs) |
| score | $s_\theta(x,\tau)=-\varepsilon_\theta/\sqrt{1-\bar\alpha}$ |
| denoiser (Tweedie) | $\hat x_0(x,\tau)=\big(x-\sqrt{1-\bar\alpha}\,\varepsilon_\theta\big)/\sqrt{\bar\alpha}$ |
| probability-flow velocity | $v_\theta(x,\tau)=-\tfrac12\beta(\tau)\big(x+s_\theta(x,\tau)\big)$, so $\dot x=v_\theta$ |
| reverse-SDE drift | $b^{\leftarrow}(x,\tau)=-\tfrac12\beta(\tau)x-\beta(\tau)s_\theta(x,\tau)$ |

The last two describe motion in forward time $\tau$: the ODE is $\mathrm dx=v_\theta\,\mathrm d\tau$ and the reverse SDE is
$\mathrm dx=b^{\leftarrow}\mathrm d\tau+\sqrt\beta\,\mathrm d\bar W$, both **integrated from $\tau=1$ down to $\tau_{\min}$**
(so a step from $\tau$ to $\tau-h$ is $x\mapsto x-h\,v_\theta$ for the ODE).

We implement the conversions. `eps_fn(x, t)` is any noise predictor: the exact `true_eps` or, later, the network.

In [ ]:
def score_from_eps(eps, x, t):
    return -eps / (1 - alpha_bar(tcol(t, x))).sqrt()

def x0_from_eps(eps, x, t):
    a = alpha_bar(tcol(t, x))
    return (x - (1 - a).sqrt() * eps) / a.sqrt()

def pf_velocity(eps_fn, x, t):
    b = beta(tcol(t, x))
    return -0.5 * b * (x + score_from_eps(eps_fn(x, t), x, t))

def reverse_drift(eps_fn, x, t):
    b = beta(tcol(t, x))
    return -0.5 * b * x - b * score_from_eps(eps_fn(x, t), x, t)

x = torch.randn(4, 2)
print("score check:", float((score_from_eps(true_eps(x, 0.2), x, 0.2) - true_score(x, 0.2)).abs().max()), "(should be ~0)")

**Remark 1.** We derive the probability-flow velocity from the Fokker–Planck equation of the forward SDE,
$\partial_\tau p=-\nabla\cdot(f p)+\tfrac12\beta\Delta p$ with $f=-\tfrac12\beta x$. Then express both $v$ and $b^\leftarrow$ through
$\varepsilon^\ast$ only.

$\tfrac12\beta\Delta p=\tfrac12\beta\nabla\cdot(p\nabla\log p)$, so
$\partial_\tau p=-\nabla\cdot\big[(f-\tfrac12\beta\nabla\log p)\,p\big]$: a continuity equation with velocity
$v=f-\tfrac12\beta\nabla\log p=-\tfrac12\beta(x+\nabla\log p)$. With $\nabla\log p=-\varepsilon^\ast/\sqrt{1-\bar\alpha}$:
$$v=-\tfrac12\beta\Big(x-\frac{\varepsilon^\ast}{\sqrt{1-\bar\alpha}}\Big),\qquad b^\leftarrow=-\tfrac12\beta\,x+\beta\frac{\varepsilon^\ast}{\sqrt{1-\bar\alpha}} .$$
What does the difference $b^\leftarrow-v=-\tfrac12\beta\nabla\log p$ do?  Integrated backward in time this is the drift $+\tfrac12\beta\nabla\log p$ that, together
with the noise $\sqrt\beta\,\mathrm d\bar W$, forms a Langevin dynamics for the current $p_\tau$: the extra drift and the extra noise
cancel in the density equation, so ODE and SDE have the same marginals.

### Looking at the exact fields
Left to right: increasing noise. Top: the score field (arrows) on the density $p_\tau$. Bottom: the denoiser — each arrow goes from
a point $x$ to $\hat x_0(x)$.

In [ ]:
g1 = torch.linspace(-3.2, 3.2, 21)
G = torch.stack(torch.meshgrid(g1, g1, indexing="xy"), -1).reshape(-1, 2)
gf = torch.linspace(-3.5, 3.5, 200)
GF = torch.stack(torch.meshgrid(gf, gf, indexing="xy"), -1).reshape(-1, 2)
fig, ax = plt.subplots(2, 3, figsize=(13, 8.6))
for j, tau in enumerate([0.05, 0.3, 0.7]):
    dens = true_log_p(GF, tau).exp().reshape(200, 200)
    s = true_score(G, tau)
    ax[0, j].contourf(gf, gf, dens, levels=20, cmap="Blues")
    ax[0, j].quiver(G[:, 0], G[:, 1], s[:, 0], s[:, 1], color="k", scale=None, width=0.003)
    ax[0, j].set_title(f"score, tau = {tau}")
    xh = x0_from_eps(true_eps(G, tau), G, tau)
    ax[1, j].contourf(gf, gf, dens, levels=20, cmap="Blues")
    ax[1, j].quiver(G[:, 0], G[:, 1], (xh - G)[:, 0], (xh - G)[:, 1], angles="xy", scale_units="xy", scale=1,
                    color="C3", width=0.003)
    ax[1, j].scatter(*CENTERS.T, c="k", marker="x")
    ax[1, j].set_title(f"denoiser x -> x0_hat(x), tau = {tau}")
for a_ in ax.flat:
    a_.set(xlim=(-3.5, 3.5), ylim=(-3.5, 3.5)); a_.set_aspect("equal")
plt.tight_layout(); plt.show()

**Question 2 (2 pts).** Look at the denoiser at small and at large noise: $\hat x_0=\sum_kw_k(x)\,\mathbb E[x_0\mid x_\tau,k]$, where for one component, $\mathbb E[x_0\mid x_\tau,k]=c_k+\frac{\sqrt{\bar\alpha}s^2}{v_\tau}(x-\sqrt{\bar\alpha}c_k)$ (Gaussian
regression). Explain why
at large $\tau$ all arrows point to (almost) the same place. Which place? Is $\hat x_0(x)$ a plausible data point there?

*Your answer:*

## 4. Training the noise predictor

The network is the MLP with a sinusoidal embedding of the continuous time $\tau$.

The loss `eps_loss`: draw $\tau\sim U(\tau_{\min},1)$, noise the batch, and return $\mathbb E\|\varepsilon-\varepsilon_\theta(x_\tau,\tau)\|^2$
(mean over the batch and the 2 coordinates).

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim=64, max_freq=1000.0):
        super().__init__()
        self.register_buffer("freqs", torch.exp(torch.linspace(0, math.log(max_freq), dim // 2)))
    def forward(self, t):                       # t: (B,)
        a = t[:, None] * self.freqs[None]
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class EpsNet(nn.Module):
    def __init__(self, hidden=256, tdim=64):
        super().__init__()
        self.emb = TimeEmbedding(tdim)
        self.net = nn.Sequential(nn.Linear(2 + tdim, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, 2))
    def forward(self, x, t):                    # x: (B, 2), t: (B,) or (B, 1)
        return self.net(torch.cat([x, self.emb(t.reshape(-1))], -1))

def eps_loss(model, x0):
    t = TAU_MIN + (1 - TAU_MIN) * torch.rand(x0.shape[0], 1)
    x_t, eps = q_sample(x0, t)
    return ((model(x_t, t) - eps) ** 2).mean()

In [ ]:
def train(steps=8000, batch=1024, lr=2e-3, ema_decay=0.999):
    model = EpsNet()
    ema = copy.deepcopy(model)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    losses, t0 = [], time.time()
    for it in range(steps):
        loss = eps_loss(model, sample_mixture(batch))      # fresh data every step
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        with torch.no_grad():
            for pe, p in zip(ema.parameters(), model.parameters()):
                pe.mul_(ema_decay).add_(p, alpha=1 - ema_decay)
        losses.append(loss.item())
        if it % 1000 == 0:
            print(f"step {it:5d}  loss {np.mean(losses[-200:]):.4f}  ({time.time() - t0:.0f}s)")
    return ema.eval(), losses

net, losses = train()
plt.figure(figsize=(5, 3)); plt.plot(np.convolve(losses, np.ones(100) / 100, "valid")); plt.xlabel("step"); plt.ylabel("loss")
plt.show()

def net_eps(x, t):
    """The trained network as a noise predictor with the same signature as true_eps."""
    return net(x, tcol(t, x))

The loss does not go to zero. What is its smallest possible value, in terms of $\varepsilon^\ast$? For each $\tau$, $\mathbb E\|\varepsilon-f(x_\tau)\|^2$ is minimized by $f=\varepsilon^\ast=\mathbb E[\varepsilon\mid x_\tau]$, and the minimum
is the conditional variance $\mathbb E\|\varepsilon-\varepsilon^\ast(x_\tau)\|^2$: the part of the noise that cannot be recovered from
$x_\tau$. Averaged over $\tau$ (divided by 2 for the per-coordinate mean) it is the floor printed below; a well-trained network ends
slightly above it.

In [ ]:
with torch.no_grad():
    xb = sample_mixture(200000)
    t = TAU_MIN + (1 - TAU_MIN) * torch.rand(xb.shape[0], 1)
    xt, e = q_sample(xb, t)
    floor = ((true_eps(xt, t) - e) ** 2).mean().item()
    final = ((net_eps(xt, t) - e) ** 2).mean().item()
print(f"best possible loss (exact eps*): {floor:.4f}   trained network: {final:.4f}")

### How good is the network?
The error $\|\varepsilon_\theta-\varepsilon^\ast\|$ on a grid, with the density $p_\tau$ as contours.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
with torch.no_grad():
    for a_, tau in zip(ax, [0.02, 0.15, 0.5]):
        err = (net_eps(GF, tau) - true_eps(GF, tau)).norm(dim=1).reshape(200, 200)
        im = a_.imshow(err, origin="lower", extent=(-3.5, 3.5, -3.5, 3.5), cmap="magma", vmin=0, vmax=0.5)
        a_.contour(gf, gf, true_log_p(GF, tau).exp().reshape(200, 200), levels=6, colors="w", linewidths=0.6)
        a_.set_title(f"|eps_theta - eps*|, tau = {tau}")
plt.colorbar(im, ax=ax, shrink=0.8); plt.show()

**Question 3 (2 pts).** Where is the network most wrong? Why does the loss not see these regions, and why does it matter little for
sampling (but not for nothing)?

*Your answer:*

## 5. Three samplers

All samplers run on the grid $1=\tau_0>\tau_1>\dots>\tau_N=\tau_{\min}$ and finish with one denoising step $x\mapsto\hat x_0(x,\tau_{\min})$.

**DDIM / DDPM step** (lecture, with $a_s=\bar\alpha(\tau_s)$ at the current time and $a_t=\bar\alpha(\tau_t)$ at the next, smaller time):
$$x_t=\sqrt{a_t}\,\hat x_0+\sqrt{1-a_t-\sigma^2}\;\varepsilon_\theta+\sigma z,\qquad
\sigma^2=\eta^2\,\frac{1-a_t}{1-a_s}\Big(1-\frac{a_s}{a_t}\Big),\quad z\sim\mathcal N(0,I),$$
where $\hat x_0$ and $\varepsilon_\theta$ are evaluated at $(x_s,\tau_s)$. $\eta=1$ is DDPM ancestral sampling, $\eta=0$ is DDIM.

Below is the `ddim_step`.

In [ ]:
def ddim_step(eps_fn, x, ts, tt, eta):
    """One step from time ts down to tt < ts."""
    a_s, a_t = alpha_bar(torch.tensor(ts)), alpha_bar(torch.tensor(tt))
    eps = eps_fn(x, ts)
    x0_hat = x0_from_eps(eps, x, ts)
    sigma2 = eta ** 2 * (1 - a_t) / (1 - a_s) * (1 - a_s / a_t)
    z = torch.randn_like(x)
    return a_t.sqrt() * x0_hat + (1 - a_t - sigma2).clamp(min=0).sqrt() * eps + sigma2.sqrt() * z

**Remark 2.** We show that for one small step ($a_s/a_t=\alpha$, the per-step factor of DDPM) the $\eta=1$ variance is the DDPM posterior
variance $\hat\beta$, and that the mean is the DDPM mean. (Use the lecture's posterior $q(x_{t-1}\mid x_t,x_0)$.)

With $a_s=\bar\alpha_t$, $a_t=\bar\alpha_{t-1}$, $a_s/a_t=\alpha_t=1-\tilde\beta_t$:
$\sigma^2=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\tilde\beta_t=\hat\beta_t$. For the mean, substitute $\varepsilon_\theta=(x_s-\sqrt{a_s}\hat x_0)/\sqrt{1-a_s}$:
the coefficient of $\hat x_0$ is $\sqrt{\bar\alpha_{t-1}}-\sqrt{1-\bar\alpha_{t-1}-\hat\beta_t}\sqrt{\bar\alpha_t}/\sqrt{1-\bar\alpha_t}$ and that of $x_s$ is
$\sqrt{1-\bar\alpha_{t-1}-\hat\beta_t}/\sqrt{1-\bar\alpha_t}$. Using $1-\bar\alpha_{t-1}-\hat\beta_t=\alpha_t(1-\bar\alpha_{t-1})^2/(1-\bar\alpha_t)$ these are
$\frac{\sqrt{\bar\alpha_{t-1}}\tilde\beta_t}{1-\bar\alpha_t}$ and $\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}$: the coefficients of $\hat\mu_t$.

**Continuous-time samplers.** The reverse SDE by Euler–Maruyama (backward step $h>0$):
$$x_{\tau-h}=x_\tau-h\,b^\leftarrow(x_\tau,\tau)+\sqrt{\beta(\tau)h}\;z,$$
and the probability-flow ODE by the standard second-order Runge–Kutta method (RK2: an Euler predictor followed by the
trapezoidal rule):
$$\tilde x=x_\tau-h\,v(x_\tau,\tau),\qquad x_{\tau-h}=x_\tau-\tfrac h2\big(v(x_\tau,\tau)+v(\tilde x,\tau-h)\big).$$

Both steps are implemented in `sde_step` and `ode_step`.


In [ ]:
def sde_step(eps_fn, x, ts, tt):
    h = ts - tt
    z = torch.randn_like(x)
    return x - h * reverse_drift(eps_fn, x, ts) + math.sqrt(float(beta(torch.tensor(ts))) * h) * z

def ode_step(eps_fn, x, ts, tt):
    h = ts - tt
    v1 = pf_velocity(eps_fn, x, ts)
    x_tilde = x - h * v1
    return x - 0.5 * h * (v1 + pf_velocity(eps_fn, x_tilde, tt))

In [ ]:
def time_grid(N, t_start=1.0):
    return np.linspace(t_start, TAU_MIN, N + 1)

@torch.no_grad()
def sample(method, eps_fn, x, N, eta=0.0, t_start=1.0, keep_path=False):
    """method in {'ddim', 'sde', 'ode' (RK2), 'euler'}; x: starting points at time t_start. Returns final samples (and the path)."""
    g = time_grid(N, t_start)
    path = [x]
    for ts, tt in zip(g[:-1], g[1:]):
        ts, tt = float(ts), float(tt)
        if method == "ddim":
            x = ddim_step(eps_fn, x, ts, tt, eta)
        elif method == "sde":
            x = sde_step(eps_fn, x, ts, tt)
        elif method == "euler":                               # Euler for the ODE in (tau, x), for comparison
            x = x - (ts - tt) * pf_velocity(eps_fn, x, ts)
        else:
            x = ode_step(eps_fn, x, ts, tt)
        path.append(x)
    x = x0_from_eps(eps_fn(x, TAU_MIN), x, TAU_MIN)          # final denoising
    path.append(x)
    return (x, torch.stack(path), np.r_[g, 0.0]) if keep_path else x

# quick look: samples from all three with the network
x1 = torch.randn(3000, 2)
fig, ax = plt.subplots(1, 4, figsize=(15, 3.8))
for a_, (name, kw) in zip(ax, [("data", None), ("ddim", dict(N=1000, eta=1.0)), ("sde", dict(N=1000)), ("ode", dict(N=200))]):
    xs = sample_mixture(3000) if kw is None else sample(name, net_eps, x1, **kw)
    a_.scatter(*xs.T, s=1, alpha=0.4); a_.set(title=name if name != "ddim" else "DDPM (ddim, eta=1)", xlim=(-3, 3), ylim=(-3, 3))
    a_.set_aspect("equal")
plt.tight_layout(); plt.show()

## 6. Watching particles

Twelve starting points $x_1\sim\mathcal N(0,I)$ are fixed. From each we run the ODE (fine grid, the "true" deterministic path), the
reverse SDE (three independent runs), DDPM ancestral sampling (= DDIM with $\eta=1$, 1000 steps, three runs), and DDIM with only
10 steps.

In [ ]:
torch.manual_seed(3)
starts = torch.randn(12, 2)
_, P_ode, g_ode = sample("ode", net_eps, starts, N=400, keep_path=True)
_, P_ddim, g_ddim = sample("ddim", net_eps, starts, N=10, eta=0.0, keep_path=True)
P_sde = [sample("sde", net_eps, starts, N=1000, keep_path=True)[1] for _ in range(3)]
P_ddpm = [sample("ddim", net_eps, starts, N=1000, eta=1.0, keep_path=True)[1] for _ in range(3)]

colors = plt.cm.tab20(np.linspace(0, 1, 12))
fig, ax = plt.subplots(1, 3, figsize=(19, 6))
for i in range(12):
    ax[0].plot(*P_ode[:, i].T, color=colors[i], lw=1.5)
    ax[0].plot(*P_ddim[:, i].T, "o--", color=colors[i], ms=4, lw=0.8)
    for P in P_sde:
        ax[1].plot(*P[:, i].T, color=colors[i], lw=0.4, alpha=0.7)
    for P in P_ddpm:
        ax[2].plot(*P[:, i].T, color=colors[i], lw=0.4, alpha=0.7)
    for a_ in ax:
        a_.scatter(*starts[i], color=colors[i], s=40, edgecolor="k", zorder=5)
ax[0].set_title("probability-flow ODE (lines) and DDIM, 10 steps (dots)")
ax[1].set_title("reverse SDE (Euler-Maruyama): 3 runs from each start")
ax[2].set_title("DDPM ancestral sampling: 3 runs from each start")
for a_ in ax:
    a_.scatter(*CENTERS.T, c="k", marker="x", s=60, zorder=6)
    a_.set(xlim=(-3, 3), ylim=(-3, 3)); a_.set_aspect("equal")
plt.tight_layout(); plt.show()

In [ ]:
g_1000 = np.r_[time_grid(1000), 0.0]
fig = plt.figure(figsize=(16, 12))
for k, (P, g, title, lw) in enumerate([(P_ode, g_ode, "probability-flow ODE (RK2, 400 steps)", 1.2),
                                       (P_ddim, g_ddim, "DDIM, 10 steps", 1.2),
                                       (P_sde[0], g_1000, "reverse SDE (one run per start)", 0.5),
                                       (P_ddpm[0], g_1000, "DDPM ancestral (one run per start)", 0.5)]):
    plot_paths_3d(fig.add_subplot(2, 2, k + 1, projection="3d"), P, g, title, colors=colors, lw=lw)
plt.tight_layout(); plt.show()

**Remark 3.** (a) Why can two ODE trajectories never cross, while SDE trajectories cross all the time?  The ODE has a smooth (locally Lipschitz) velocity, so through each point $(x,\tau)$ passes exactly one solution:
two trajectories that met would coincide forever. The SDE has a fresh random increment at every instant, so paths are not determined
by their current position.

(b) Each SDE run from the same
start may end in a different mode; each ODE run ends in the same place. Yet both produce the correct distribution from
$x_1\sim\mathcal N(0,I)$. How is the randomness distributed in each case?  For the ODE all randomness is in the starting point: the map $x_1\mapsto x_0$ is a deterministic
transport of $\mathcal N(0,I)$ to the data, and regions of the start space are assigned to modes. For the SDE the randomness is spread
along the path; the start only matters a little (by $\tau=1$ almost all memory of the data is gone).

(c) Where do the DDIM dots lie, and why? On (or very near) the ODE
trajectories: DDIM with $\eta=0$ is an Euler step of the same ODE in the variables $y=x/\sqrt{\bar\alpha}$, $\sigma=\sqrt{(1-\bar\alpha)/\bar\alpha}$,
where the trajectories are nearly straight over most of the noise range, so even 10 steps stay close (the visible deviations
appear where the paths bend, at small noise).

### Starting in the middle: where does the SDE take one point?
Fix one point $x$ at $\tau_0=0.4$ and run the reverse SDE from it many times.

In [ ]:
tau0 = 0.4
xstart = torch.tensor([[0.3, 0.5]])
ends = sample("sde", net_eps, xstart.repeat(20000, 1), N=600, t_start=tau0)
ode_end = sample("ode", net_eps, xstart, N=300, t_start=tau0)
with torch.no_grad():
    tweedie = x0_from_eps(net_eps(xstart, tau0), xstart, tau0)
    tweedie_exact = x0_from_eps(true_eps(xstart, tau0), xstart, tau0)
print("mean of SDE endpoints :", ends.mean(0).numpy().round(3), " +-", (ends.std(0) / math.sqrt(len(ends))).numpy().round(3))
print("Tweedie x0_hat (net)   :", tweedie[0].numpy().round(3), "  (exact:", tweedie_exact[0].numpy().round(3), ")")
print("ODE endpoint          :", ode_end[0].numpy().round(3))
plt.figure(figsize=(5.5, 5.5))
plt.scatter(*ends[:3000].T, s=1, alpha=0.3, label="SDE endpoints")
plt.scatter(*xstart.T, c="k", s=60, label=f"start x at tau={tau0}")
plt.scatter(*ends.mean(0), c="C3", s=120, marker="*", label="mean of endpoints")
plt.scatter(*tweedie.T, c="C2", s=60, marker="D", label="Tweedie x0_hat")
plt.scatter(*ode_end.T, c="C1", s=80, marker="s", label="ODE endpoint")
plt.legend(fontsize=8); plt.gca().set_aspect("equal"); plt.xlim(-3, 3); plt.ylim(-3, 3); plt.show()

_, P_fix, g_fix = sample("sde", net_eps, xstart.repeat(40, 1), N=600, t_start=tau0, keep_path=True)
_, P_fix_ode, g_fix_ode = sample("ode", net_eps, xstart, N=300, t_start=tau0, keep_path=True)
fig = plt.figure(figsize=(12, 6))
a3 = fig.add_subplot(111, projection="3d")
plot_paths_3d(a3, P_fix, g_fix, f"40 SDE runs from one point at tau = {tau0} (black: the ODE path)",
              slices=(tau0, 0.2, 0.0), start_label=f"x at tau={tau0}", end_label="SDE endpoints", lw=0.5)
a3.plot(g_fix_ode, P_fix_ode[:, 0, 0], P_fix_ode[:, 0, 1], color="k", lw=2.5)
plt.tight_layout(); plt.show()

**Remark 4.** (a) What distribution do the SDE endpoints follow? The reverse SDE reproduces the law of the forward process backwards, including its conditionals: started from
$x_{\tau_0}=x$ it samples $q(x_0\mid x_{\tau_0}=x)$, the posterior of the data given the noisy point (here: mostly the nearby modes,
weighted by $w_k(x)$).

(b) Why their mean agrees with Tweedie's $\hat x_0(x,\tau_0)$. The mean of that posterior is $\mathbb E[x_0\mid x_{\tau_0}=x]$, which is exactly Tweedie's denoiser — the network
predicts the *average* of all the places the stochastic dynamics can end.

(c) Why is the ODE endpoint *not* at $\hat x_0$? The ODE is deterministic: it sends $x$ to one point of
the data distribution (on one mode), chosen so that the *population* is transported correctly. A single point cannot be the posterior
mean, which lies between modes.

### DDIM converges to the ODE
The deterministic DDIM map $x_1\mapsto x_0$ should converge to the ODE map as the number of steps grows. We compare it with two
standard integrators of the same ODE in the original variables $(\tau,x)$: Euler and RK2.

In [ ]:
torch.manual_seed(5)
xs1 = torch.randn(2000, 2)
ref = sample("ode", net_eps, xs1, N=1000)
Ns = [5, 10, 20, 50, 100, 200]
dist = {m: [(sample(m, net_eps, xs1, N=n) - ref).norm(dim=1).mean().item() for n in Ns] for m in ["ddim", "euler", "ode"]}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for m, lab, cost in [("ddim", "DDIM (eta=0)", 1), ("euler", "Euler in (tau, x)", 1), ("ode", "RK2 in (tau, x)", 2)]:
    ax[0].loglog(Ns, dist[m], "o-", label=lab)
    ax[1].loglog([cost * n for n in Ns], dist[m], "o-", label=lab)
ax[0].set(xlabel="steps N", ylabel="mean distance to the ODE endpoint"); ax[1].set(xlabel="network evaluations")
ax[0].legend(); plt.tight_layout(); plt.show()

**Remark 5.** The convergence order of each method (left). Each RK2 step costs two network evaluations, DDIM and Euler are both
first order and both deterministic: in what sense are they different methods for the same ODE?

Slopes $\approx-1$ for DDIM and Euler (first order: error $\propto1/N$) and $\approx-2$ for RK2 (second
order). Per evaluation, RK2 wins once the budget is moderate (here beyond roughly 20 evaluations); with very few evaluations a
first-order method can be as good or better, because the second-order correction is only accurate once steps are small.
DDIM is Euler's method applied to the same ODE written in other variables, $y=x/\sqrt{\bar\alpha}$ and $\sigma=\sqrt{(1-\bar\alpha)/\bar\alpha}$,
where it reads $\mathrm dy/\mathrm d\sigma=\varepsilon^\ast$. Both have error $\propto1/N$ but with different constants: DDIM is exact when
$\varepsilon^\ast$ is constant along a path (straight paths in $(\sigma,y)$), Euler in $(\tau,x)$ is not even then. Which constant is smaller
depends on the data and on the time grid; on this two-dimensional mixture with a grid uniform in $\tau$ the two are of the same size
(Euler may even be slightly better), whereas on images DDIM is far better than naive Euler.

## 7. Accuracy against cost: sampler error vs network error

We measure the distance between generated samples and exact data samples (sliced $W_1$ Kantorovich distance, 32 directions, $2\cdot10^4$ points)
for each sampler and number of steps, once with the **network** and once with the **exact** $\varepsilon^\ast$. The difference between
the two curves is the error of the network; the exact curve is the error of the sampler alone.

In [ ]:
PROJ = torch.stack([torch.cos(torch.linspace(0, math.pi, 33)[:-1]), torch.sin(torch.linspace(0, math.pi, 33)[:-1])], 1)
def sliced_w1(a, b):
    pa, pb = torch.sort(a @ PROJ.T, 0)[0], torch.sort(b @ PROJ.T, 0)[0]
    return (pa - pb).abs().mean().item()

n = 20000
data_ref = sample_mixture(n)
print("floor (two sets of exact samples):", round(sliced_w1(sample_mixture(n), data_ref), 4))
settings = {"DDPM (eta=1)": ("ddim", 1.0, 1), "DDIM (eta=0)": ("ddim", 0.0, 1), "ODE RK2": ("ode", 0.0, 2), "SDE EM": ("sde", 0.0, 1)}
Ns = [5, 10, 20, 50, 100, 300]
results = {}
for which, fn in [("network", net_eps), ("exact", true_eps)]:
    for name, (m, eta, cost) in settings.items():
        errs = []
        for N in Ns:
            torch.manual_seed(N)
            errs.append(sliced_w1(sample(m, fn, torch.randn(n, 2), N=N, eta=eta), data_ref))
        results[(which, name)] = ([cost * N for N in Ns], errs)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a_, which in zip(ax, ["exact", "network"]):
    for name in settings:
        nfe, errs = results[(which, name)]
        a_.loglog(nfe, errs, "o-", label=name)
    a_.set(title=f"{which} eps", xlabel="network evaluations (NFE)"); a_.legend(fontsize=8)
ax[0].set_ylabel("sliced W1 to the data"); plt.tight_layout(); plt.show()

**Remark 6.** (a) *Which sampler needs the fewest evaluations for a given accuracy with the exact $\varepsilon^\ast$?* With the exact $\varepsilon^\ast$, RK2 for the ODE reaches a given
error with the fewest evaluations once the budget is moderate; at 5–10 steps the stochastic samplers (DDPM, reverse SDE) are the
worst. With many steps all samplers reach the measurement floor; the reverse SDE catches up surprisingly early, because its noise
keeps pulling particles back toward $p_\tau$ and so corrects earlier errors.

(b) *Where do the curves with the network flatten out, and what sets that level?* With the network the curves flatten at the level of
the network error: once the discretization error is below it, more steps cannot help.

(c) *Why does the reverse SDE need many more steps than the ODE?* The SDE drift contains $\beta\nabla\log p$,
twice the score term of the ODE, which is large and fast-varying at small $\tau$, and each step adds noise of size $\sqrt{\beta h}$ that
later steps must remove; with few large steps the Gaussian steps also misrepresent the non-Gaussian reverse kernels.

(d) *Which sampler would you use with 20 evaluations?* A deterministic sampler: DDIM, or RK2 with about 10 steps.

## 8. Guidance: steering the sampler toward a class

Give each data point a label $y\in\{0,\dots,7\}$: the component it was drawn from. We want samples of one class $y$, and possibly
"more typical" ones. Bayes' rule splits the conditional score into the unconditional score and a classifier term:
$$\nabla_x\log p_\tau(x\mid y)=\nabla_x\log p_\tau(x)+\nabla_x\log p_\tau(y\mid x).$$
**Guidance** multiplies the classifier term by a scale $\gamma\ge0$:
$$\tilde s(x,\tau)=s_\theta(x,\tau)+\gamma\,\nabla_x\log p_\tau(y\mid x)
\quad\Longleftrightarrow\quad
\tilde\varepsilon=\varepsilon_\theta-\gamma\sqrt{1-\bar\alpha}\,\nabla_x\log p_\tau(y\mid x),$$
and then runs any of our samplers with $\tilde\varepsilon$ in place of $\varepsilon_\theta$ ($\gamma=0$: unconditional; $\gamma=1$: the class
conditional). We get the classifier in three ways:

1. **Exactly, by Bayes' rule.** For our mixture, $p_\tau(y=k\mid x)\propto\frac1K\mathcal N(x;\sqrt{\bar\alpha}c_k,v_\tau I)$.
2. **A trained noisy classifier** $p_\phi(y\mid x_\tau,\tau)$, as in classifier guidance (Dhariwal & Nichol 2021).
3. **No classifier at all:** classifier-free guidance (Ho & Salimans 2022), where the classifier is implicit in a conditional
   diffusion model.

In [ ]:
def sample_mixture_labeled(n):
    y = torch.randint(K, (n,))
    return CENTERS[y] + STD * torch.randn(n, 2), y

def as_labels(y, x):
    y = torch.as_tensor(y)
    return y.expand(x.shape[0]) if y.ndim == 0 else y

### 8.1 The exact classifier

We provide `true_class_log_prob`, the $(B,K)$ matrix of $\log p_\tau(y=k\mid x)$. (Use `_mixture_terms`; all components have the
same variance and the same prior weight.)

and `guided_eps`, which returns a new noise predictor $\tilde\varepsilon$ built from a noise predictor, a classifier and
a scale. The gradient $\nabla_x\log p_\tau(y\mid x)$ is provided by `classifier_grad`.

In [ ]:
def true_class_log_prob(x, t):
    a, v, m, d2 = _mixture_terms(x, t)
    return torch.log_softmax(-d2 / (2 * v), dim=1)

def classifier_grad(log_prob_fn, x, t, y):
    """grad_x log p_tau(y | x) for every sample (works inside torch.no_grad())."""
    with torch.enable_grad():
        x = x.detach().requires_grad_(True)
        lp = log_prob_fn(x, t)[torch.arange(x.shape[0]), as_labels(y, x)].sum()
        return torch.autograd.grad(lp, x)[0]

def guided_eps(eps_fn, log_prob_fn, y, gamma):
    def eps_tilde(x, t):
        a = alpha_bar(tcol(t, x))
        return eps_fn(x, t) - gamma * (1 - a).sqrt() * classifier_grad(log_prob_fn, x, t, y)
    return eps_tilde

def true_cond_eps(x, t, y):
    """Exact noise prediction of the class-conditional density p_tau(x | y): one Gaussian component."""
    a, v, m, d2 = _mixture_terms(x, t)
    mk = m[torch.arange(x.shape[0]), as_labels(y, x)]
    return -(1 - a).sqrt() * (mk - x) / v

x = torch.randn(6, 2)
e1 = guided_eps(true_eps, true_class_log_prob, 3, 1.0)(x, 0.3)
print("gamma = 1 equals the exact conditional:", float((e1 - true_cond_eps(x, 0.3, 3)).abs().max()), "(should be ~0)")

**Remark 7.** (a) We show that $p_\tau(y=k\mid x)=w_k(x)$, the weights of Section 2, and that $\nabla_x\log p_\tau(y=k\mid x)=s_k(x)-\nabla\log p_\tau(x)$,
where $s_k=(\sqrt{\bar\alpha}c_k-x)/v_\tau$ is the score of component $k$.

(b) With the exact score and $\gamma=1$ the guided
score is the exact conditional score (the check printed above).

(c) The classifier depends on $\tau$. Why can't we use one classifier
trained on clean data $x_0$?

**Solution.** (a) Bayes: $p_\tau(k\mid x)=\frac{\frac1K\mathcal N_k(x)}{\sum_j\frac1K\mathcal N_j(x)}=w_k(x)$. Hence
$\log p_\tau(k\mid x)=\log\mathcal N_k(x)-\log\sum_j\mathcal N_j(x)$, whose gradient is $s_k-\nabla\log p_\tau$ (the second term is
$\log p_\tau$ up to a constant).

(b) $\nabla\log p_\tau+1\cdot(s_k-\nabla\log p_\tau)=s_k=\nabla\log p_\tau(x\mid k)$.

(c) The sampler needs
$\nabla\log p_\tau(y\mid x_\tau)$ for *noisy* inputs: the probability of the class given a noisy point. A clean-data classifier answers
a different question, is never trained on noisy inputs, and its gradients there are meaningless.

### 8.2 A trained noisy classifier

We complete the loss: noise the labelled batch at random times and return the cross-entropy of the classifier's logits.

In [ ]:
class NoisyClassifier(nn.Module):
    def __init__(self, hidden=128, tdim=64):
        super().__init__()
        self.emb = TimeEmbedding(tdim)
        self.net = nn.Sequential(nn.Linear(2 + tdim, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, K))
    def forward(self, x, t):
        return self.net(torch.cat([x, self.emb(t.reshape(-1))], -1))

def classifier_loss(clf, x0, y):
    t = TAU_MIN + (1 - TAU_MIN) * torch.rand(x0.shape[0], 1)
    x_t, _ = q_sample(x0, t)
    return nn.functional.cross_entropy(clf(x_t, t), y)

clf = NoisyClassifier()
opt = torch.optim.Adam(clf.parameters(), lr=2e-3)
for it in range(4000):
    xb, yb = sample_mixture_labeled(1024)
    loss = classifier_loss(clf, xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 1000 == 0:
        print(f"step {it}  cross-entropy {loss.item():.3f}")
clf.eval()

def clf_log_prob(x, t):
    return torch.log_softmax(clf(x, tcol(t, x)), dim=1)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 7.6))
with torch.no_grad():
    for j, tau in enumerate([0.05, 0.3, 0.6]):
        for i, (fn, name) in enumerate([(true_class_log_prob, "Bayes (exact)"), (clf_log_prob, "trained classifier")]):
            pr = fn(GF, tau)[:, 0].exp().reshape(200, 200)
            im = ax[i, j].imshow(pr, origin="lower", extent=(-3.5, 3.5, -3.5, 3.5), vmin=0, vmax=1, cmap="viridis")
            ax[i, j].scatter(*CENTERS.T, c="w", marker="x", s=20)
            ax[i, j].set_title(f"{name}: p(y=0 | x), tau={tau}", fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8); plt.show()

**Question 4 (2 pts).** Compare the two rows. Where does the trained classifier disagree with Bayes' rule, and why there? Its *gradient* is
what guidance uses: which kind of error matters more, a wrong probability or a wrong slope?

*Your answer:*

### 8.3 Classifier-free guidance

Train **one** network $\varepsilon_\theta(x,\tau,c)$ with a class input $c\in\{0,\dots,7\}$ and an extra "null" class $c=\varnothing$ (index 8).
During training the label is replaced by $\varnothing$ with probability $p_{\text{uncond}}=0.2$, so the same network learns both the
conditional and the unconditional noise predictor. At sampling time
$$\tilde\varepsilon=\varepsilon_\theta(x,\tau,\varnothing)+w\,\big(\varepsilon_\theta(x,\tau,y)-\varepsilon_\theta(x,\tau,\varnothing)\big).$$

In `cfg_loss` each label is replaced by `NULL` with probability `p_uncond`; `cfg_eps` forms the guided combination.

In [ ]:
NULL = K

class CondEpsNet(nn.Module):
    def __init__(self, hidden=256, tdim=64, cdim=32):
        super().__init__()
        self.emb = TimeEmbedding(tdim)
        self.cls = nn.Embedding(K + 1, cdim)
        self.net = nn.Sequential(nn.Linear(2 + tdim + cdim, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, 2))
    def forward(self, x, t, c):
        return self.net(torch.cat([x, self.emb(t.reshape(-1)), self.cls(c)], -1))

def cfg_loss(model, x0, y, p_uncond=0.2):
    t = TAU_MIN + (1 - TAU_MIN) * torch.rand(x0.shape[0], 1)
    drop = torch.rand(y.shape[0]) < p_uncond
    c = torch.where(drop, torch.full_like(y, NULL), y)
    x_t, eps = q_sample(x0, t)
    return ((model(x_t, t, c) - eps) ** 2).mean()

def cfg_eps(model, y, w):
    def eps_tilde(x, t):
        tt, c = tcol(t, x), as_labels(y, x)
        e_uncond = model(x, tt, torch.full_like(c, NULL))
        e_cond = model(x, tt, c)
        return e_uncond + w * (e_cond - e_uncond)
    return eps_tilde

cnet = CondEpsNet(); cema = copy.deepcopy(cnet)
opt = torch.optim.Adam(cnet.parameters(), lr=2e-3); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 8000)
for it in range(8000):
    xb, yb = sample_mixture_labeled(1024)
    loss = cfg_loss(cnet, xb, yb)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    with torch.no_grad():
        for pe, p in zip(cema.parameters(), cnet.parameters()):
            pe.mul_(0.999).add_(p, alpha=0.001)
    if it % 2000 == 0:
        print(f"step {it}  loss {loss.item():.4f}")
cnet = cema.eval()

**Remark 8.** (a) CFG combination corresponds to the score $s(x)+w\,\big(s(x\mid y)-s(x)\big)=s(x)+w\,\nabla_x\log p_\tau^{\,\mathrm{impl}}(y\mid x)$,
i.e. classifier guidance with an *implicit* classifier $p^{\mathrm{impl}}(y\mid x)\propto p(x\mid y)/p(x)$.

(b) With the exact noise predictors, CFG with scale $w$ and exact-Bayes classifier guidance with $\gamma=w$ give the same sampler. Why?

(c) Give two practical reasons why CFG replaced classifier guidance.

**Solution.** (a) $\varepsilon=-\sqrt{1-\bar\alpha}\,s$ is linear, so $\tilde\varepsilon=\varepsilon_\varnothing+w(\varepsilon_y-\varepsilon_\varnothing)$ corresponds to
$\tilde s=s+w(s_y-s)$, and $s_y-s=\nabla\log p(x\mid y)-\nabla\log p(x)=\nabla\log\frac{p(x\mid y)}{p(x)}=\nabla\log p(y\mid x)$ by Bayes (the
prior $p(y)$ does not depend on $x$).

(b) The exact conditional and unconditional predictors give $s_y-s=s_k-s$, the exact Bayes
classifier gradient of Remark 7.

(c) No second network to train on noisy data; the "classifier" is built from the same features
as the generator, so its gradients are smooth and aligned with the data rather than adversarial; one network serves conditional,
unconditional and guided sampling; and it works with conditions that are not classes (text prompts), where training a noisy
classifier is impractical.

### 8.4 What guidance does to the samples
Class $y=0$ (the component at $(1.7,0)$), ODE sampler with 100 RK2 steps, for three guidance methods and several scales.

In [ ]:
Y_TARGET = 0
scales = [0.0, 1.0, 3.0, 8.0]
methods = {
    "classifier guidance, Bayes": lambda g: guided_eps(net_eps, true_class_log_prob, Y_TARGET, g),
    "classifier guidance, trained": lambda g: guided_eps(net_eps, clf_log_prob, Y_TARGET, g),
    "classifier-free": lambda g: cfg_eps(cnet, Y_TARGET, g),
}
torch.manual_seed(11)
xg = torch.randn(2000, 2)
fig, ax = plt.subplots(len(methods), len(scales), figsize=(14, 10.5))
for i, (mname, make) in enumerate(methods.items()):
    for j, g in enumerate(scales):
        out = sample("ode", make(g), xg, N=100)
        ax[i, j].scatter(*out.T, s=1, alpha=0.4)
        ax[i, j].scatter(*CENTERS.T, c="k", marker="x", s=25)
        ax[i, j].set(xlim=(-3, 3.5), ylim=(-3, 3), title=f"{mname}\nscale = {g}"); ax[i, j].title.set_fontsize(9)
        ax[i, j].set_aspect("equal")
plt.tight_layout(); plt.show()

### 8.5 Paths under guidance, in 2D and in 3D
The same twelve starting points as in Section 6, guided toward class 0 with the exact-Bayes classifier at $\gamma=1$ and $\gamma=6$, and
with CFG at $w=6$. ODE paths are solid; one SDE run per start (for $\gamma=1$) is thin. In 3D the faint clouds are the unconditional
marginals (noise at $\tau=1$, all data at $\tau=0$) and the orange cloud is the target class: guided particles must leave the
unconditional flow and end in the orange cloud.

In [ ]:
cases = [("Bayes CG, gamma = 1", guided_eps(net_eps, true_class_log_prob, Y_TARGET, 1.0)),
         ("Bayes CG, gamma = 6", guided_eps(net_eps, true_class_log_prob, Y_TARGET, 6.0)),
         ("CFG, w = 6", cfg_eps(cnet, Y_TARGET, 6.0))]
paths = {}
for name, fn in cases:
    paths[name] = sample("ode", fn, starts, N=300, keep_path=True)[1:]
sde_g1 = sample("sde", cases[0][1], starts, N=1000, keep_path=True)[1:]

fig, ax = plt.subplots(1, 3, figsize=(17, 5.8))
for a_, (name, _) in zip(ax, cases):
    P, g = paths[name]
    for i in range(12):
        a_.plot(*P[:, i].T, color=colors[i], lw=1.6)
        if name.endswith("= 1"):
            a_.plot(*sde_g1[0][:, i].T, color=colors[i], lw=0.3, alpha=0.6)
        a_.scatter(*starts[i], color=colors[i], s=40, edgecolor="k", zorder=5)
    a_.scatter(*CENTERS.T, c="k", marker="x", s=60, zorder=6)
    a_.set(xlim=(-3, 4.5), ylim=(-3, 3.2), title=name); a_.set_aspect("equal")
plt.tight_layout(); plt.show()

x_cls, y_cls = sample_mixture_labeled(4000)
target_cloud = x_cls[y_cls == Y_TARGET][:300]
fig = plt.figure(figsize=(16, 12))
for k, (name, _) in enumerate(cases):
    P, g = paths[name]
    plot_paths_3d(fig.add_subplot(2, 2, k + 1, projection="3d"), P, g, name + " (ODE)", slices=(1.0, 0.0),
                  highlight=target_cloud, colors=colors, lw=1.2)
plot_paths_3d(fig.add_subplot(2, 2, 4, projection="3d"), sde_g1[0], sde_g1[1], "Bayes CG, gamma = 1 (SDE)", slices=(1.0, 0.0),
              highlight=target_cloud, colors=colors, lw=0.5)
plt.tight_layout(); plt.show()

### 8.6 Is guidance sampling the "sharpened" distribution?
A common reading of $\gamma>1$: sample $p_\gamma(x\mid y)\propto p_0(x)\,p_0(y\mid x)^\gamma$, a sharpened conditional. We can compute the
statistics of $p_\gamma$ exactly by reweighting data samples by $p_0(y\mid x)^\gamma$, and compare them with what the guided ODE and SDE
(exact score, exact classifier) actually produce: the mean radius of the class-0 samples and their spread across the circle
(the $y$-coordinate's standard deviation).

In [ ]:
gammas = [0, 1, 2, 4, 8, 16]
xd = sample_mixture(200000)
lw0 = true_class_log_prob(xd, TAU_MIN)[:, Y_TARGET]
stats = {"tilted target": [], "guided ODE": [], "guided SDE": []}
torch.manual_seed(2)
xs1 = torch.randn(5000, 2)
for g in gammas:
    if g == 0:      # no tilt: the class-0 part of the data
        sel = xd[true_class_log_prob(xd, TAU_MIN).argmax(1) == Y_TARGET]
        stats["tilted target"].append((sel[:, 0].mean().item(), sel[:, 1].std().item()))
    else:           # weights proportional to p_0(y | x)^gamma, normalized over the data samples
        wts = torch.softmax(g * lw0, 0)
        mx = (wts * xd[:, 0]).sum().item()
        sy = ((wts * xd[:, 1] ** 2).sum() - (wts * xd[:, 1]).sum() ** 2).sqrt().item()
        stats["tilted target"].append((mx, sy))
    for name, m, N in [("guided ODE", "ode", 150), ("guided SDE", "sde", 1000)]:
        out = sample(m, guided_eps(true_eps, true_class_log_prob, Y_TARGET, g), xs1, N=N)
        out = out[true_class_log_prob(out, TAU_MIN).argmax(1) == Y_TARGET]
        stats[name].append((out[:, 0].mean().item(), out[:, 1].std().item()))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for name, st in stats.items():
    st = np.array(st)
    ax[0].plot(gammas, st[:, 0], "o-", label=name); ax[1].plot(gammas, st[:, 1], "o-", label=name)
ax[0].set(xlabel="gamma", ylabel="mean x-coordinate of class-0 samples"); ax[1].set(xlabel="gamma", ylabel="std across the circle")
ax[0].axhline(1.7, color="gray", lw=0.6); ax[0].legend(); plt.tight_layout(); plt.show()

**Remark 9.** (a) What increasing $\gamma$ does to the ODE samples, and to the SDE samples.

(b) Does either sample the
tilted target $p_0(x)\,p_0(y\mid x)^\gamma$?

(c) Is $\nabla\log p_\tau+\gamma\nabla\log p_\tau(y\mid x)$ the score of the noised tilted
distribution? Why do the ODE and the SDE no longer agree, although they did for every unguided model?

(a) For the ODE, larger $\gamma$ pushes the class-0 samples *outward*, away from
the other classes (the mean $x$-coordinate grows from $1.7$ to well above $2$), and squeezes them across the circle (the spread drops
far below the data's $0.2$): fewer, more "extreme" samples. The SDE samples move much less: a slight outward shift, nearly the
original spread.

(b) Neither. The tilted target barely differs from the class-0 component itself, because $p_0(y=0\mid x)\approx1$
on that component already; the guided samplers move far beyond it.

(c) No. Tilting by $p(y\mid x)^\gamma$ and adding noise do not
commute: the noised tilted density is $\int q(x_\tau\mid x_0)\,p_0(x_0)p_0(y\mid x_0)^\gamma\,\mathrm dx_0$, whose score is not
$s+\gamma\nabla\log p_\tau(y\mid x)$ for $\gamma\ne1$. The guided field is the strongest at intermediate $\tau$, where $p_\tau(y\mid x)$ is
uncertain and its gradient points away from the other classes, and it is not the score of any family of densities connected by the
forward diffusion. The equivalence of the ODE and the SDE rests on the drift difference $\frac12\beta\nabla\log p_\tau$ being the score
of the *current* density of the particles; with a field that is not that score, the two dynamics move the population differently.
The SDE's noise keeps re-spreading the particles, so it reacts less. In practice $\gamma$ (or $w$) is a knob that trades diversity
for fidelity, not a way to sample a well-defined distribution.

## 9. Exact likelihood through the ODE

The probability-flow ODE is a continuous normalizing flow. Along a trajectory $\frac{\mathrm d}{\mathrm d\tau}\log p_\tau(x_\tau)=-\nabla\cdot v(x_\tau,\tau)$, so
integrating **forward** from a data point at $\tau_{\min}$ to $\tau=1$,
$$\log p_{\tau_{\min}}(x)=\log\mathcal N(x_1;0,I)+\int_{\tau_{\min}}^1\nabla\cdot v(x_\tau,\tau)\,\mathrm d\tau .$$
In 2D the divergence can be computed exactly with two backward passes.

`divergence` computes the sum of the diagonal of the Jacobian of $v$; we compare the result with the exact
$\log p_{\tau_{\min}}$.
(`torch.autograd.grad(v[:, i].sum(), x, retain_graph=True)[0]` is the gradient of the $i$-th output with respect to $x$ for
every sample at once; we take its $i$-th column, for $i=0,1$.)

In [ ]:
def divergence(eps_fn, x, t):
    """Returns v(x, t) and div_x v(x, t), both detached."""
    x = x.detach().requires_grad_(True)
    v = pf_velocity(eps_fn, x, t)
    div = sum(torch.autograd.grad(v[:, i].sum(), x, retain_graph=True)[0][:, i] for i in range(2))
    return v.detach(), div.detach()

def ode_log_likelihood(eps_fn, x, N=400):
    g = np.linspace(TAU_MIN, 1.0, N + 1)
    logdet = torch.zeros(x.shape[0])
    for ts, tt in zip(g[:-1], g[1:]):                      # RK2, forward in time
        h = float(tt - ts)
        v1, d1 = divergence(eps_fn, x, float(ts))
        x_tilde = x + h * v1
        v2, d2 = divergence(eps_fn, x_tilde, float(tt))
        x = x + 0.5 * h * (v1 + v2)
        logdet = logdet + 0.5 * h * (d1 + d2)
    log_prior = -0.5 * (x ** 2).sum(1) - math.log(2 * math.pi)
    return log_prior + logdet

xtest = sample_mixture(300)
ll_exact_eps = ode_log_likelihood(true_eps, xtest)
ll_net = ode_log_likelihood(net_eps, xtest)
ll_true = true_log_p(xtest, TAU_MIN)
print("mean log-likelihood: true", ll_true.mean().item(), " ODE(exact eps)", ll_exact_eps.mean().item(), " ODE(network)", ll_net.mean().item())
plt.figure(figsize=(5, 5))
plt.scatter(ll_true, ll_exact_eps, s=6, label="ODE with exact eps")
plt.scatter(ll_true, ll_net, s=6, label="ODE with network")
lo, hi = ll_true.min().item(), ll_true.max().item(); plt.plot([lo, hi], [lo, hi], "k--", lw=0.8)
plt.xlabel("true log p"); plt.ylabel("log-likelihood from the ODE"); plt.legend(); plt.show()

**Question 5 (2 pts).** With the exact $\varepsilon^\ast$ the points lie on the diagonal: why is this an *exact* method (up to
discretization), unlike the ELBO of DDPM? With the network they scatter: is the network's log-likelihood an estimate of the true
$\log p$, or the exact log-density of some other model? Which model?

*Hint.* The ODE is an invertible map from $\mathcal N(0,I)$ (at $\tau=1$) to the data.

*Your answer:*